# A PySpark-Based Yelp Big Data Analytics and Business Recommendation Pipeline

## Phase 0. Introduction and Problem Framing

**Core problem**  
How can big data analytics reveal the characteristics of highly rated, highly engaged, and highly active businesses, and how can those insights be used to generate relevant business recommendations?

**What this project is about**  
In this notebook, I use four Yelp datasets together: `business.json`, `review.json`, `checkin.json`, and `tip.json`. The work is split into two connected parts.

- **Part A** is where I clean the data, build features, and analyse what makes businesses strong in terms of quality, engagement, and activity.
- **Part B** uses those processed outputs to build a recommendation system that can help people discover relevant businesses more easily.

**Why I used Yelp**  
The Yelp dataset is a good fit for this project because it does not only contain business details. It also contains review behaviour, check-in activity, and user tips. That makes it possible to connect big data analytics with recommendation in one full pipeline.


### Aim
To build a PySpark-based pipeline that analyses Yelp businesses and then uses those results to generate business recommendations.

### Objectives
1. Load and combine large Yelp datasets using PySpark.
2. Clean and structure the raw data so it can be analysed properly.
3. Study the factors linked to business quality, engagement, and activity.
4. Create enriched business-level features from multiple sources.
5. Reuse those processed outputs in a recommendation stage.
6. Build a simple popularity-based baseline.
7. Build a main content-based recommender using business similarity.


### High-Level Notebook Flow

1. **Phase 0–2**: problem framing, Spark setup, and raw ingestion of the four Yelp datasets  
2. **Part A**: cleaning, feature engineering, analytics, visualization, and processed dataset export  
3. **Part B**: load processed outputs, build a popularity baseline, and build the main content-based recommender  


## Phase 1. Environment Setup and Project Paths

In [ ]:
from pathlib import Path
import os
import platform

import findspark
findspark.init()

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def resolve_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not resolve the project root from the current notebook location.")


PROJECT_ROOT = resolve_project_root(Path.cwd())
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

BUSINESS_DATA_PATH = RAW_DATA_DIR / "yelp_academic_dataset_business.json"
REVIEW_DATA_PATH = RAW_DATA_DIR / "yelp_academic_dataset_review.json"
CHECKIN_DATA_PATH = RAW_DATA_DIR / "yelp_academic_dataset_checkin.json"
TIP_DATA_PATH = RAW_DATA_DIR / "yelp_academic_dataset_tip.json"

for required_path in [BUSINESS_DATA_PATH, REVIEW_DATA_PATH, CHECKIN_DATA_PATH, TIP_DATA_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required Yelp dataset file not found: {required_path}")

for directory in [PROCESSED_DATA_DIR, FIGURES_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"Tables directory: {TABLES_DIR}")


In [ ]:
spark = (
    SparkSession.builder
    .appName("Yelp Integrated Analytics and Recommendation Pipeline")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Python version: {platform.python_version()}")
print(f"Spark application name: {spark.sparkContext.appName}")


## Phase 2. Load the Four Raw Yelp Datasets

In [ ]:
business_raw_df = spark.read.json(str(BUSINESS_DATA_PATH))
review_raw_df = spark.read.json(str(REVIEW_DATA_PATH))
checkin_raw_df = spark.read.json(str(CHECKIN_DATA_PATH))
tip_raw_df = spark.read.json(str(TIP_DATA_PATH))

print("Raw Yelp datasets loaded successfully.")


In [ ]:
dataset_summary = [
    {
        "dataset": "business",
        "rows": business_raw_df.count(),
        "columns": len(business_raw_df.columns),
        "size_mb": round(BUSINESS_DATA_PATH.stat().st_size / (1024 ** 2), 2),
        "role": "business metadata and item features",
    },
    {
        "dataset": "review",
        "rows": review_raw_df.count(),
        "columns": len(review_raw_df.columns),
        "size_mb": round(REVIEW_DATA_PATH.stat().st_size / (1024 ** 2), 2),
        "role": "explicit user-business interaction and engagement",
    },
    {
        "dataset": "checkin",
        "rows": checkin_raw_df.count(),
        "columns": len(checkin_raw_df.columns),
        "size_mb": round(CHECKIN_DATA_PATH.stat().st_size / (1024 ** 2), 2),
        "role": "business activity and popularity behavior",
    },
    {
        "dataset": "tip",
        "rows": tip_raw_df.count(),
        "columns": len(tip_raw_df.columns),
        "size_mb": round(TIP_DATA_PATH.stat().st_size / (1024 ** 2), 2),
        "role": "lightweight interaction and supplementary engagement",
    },
]

pd.DataFrame(dataset_summary)


In [ ]:
print("Business schema")
business_raw_df.printSchema()

print("\nReview schema")
review_raw_df.printSchema()

print("\nCheck-in schema")
checkin_raw_df.printSchema()

print("\nTip schema")
tip_raw_df.printSchema()


In [ ]:
print("Business sample")
business_raw_df.show(3, truncate=False)

print("\nReview sample")
review_raw_df.show(3, truncate=False)

print("\nCheck-in sample")
checkin_raw_df.show(3, truncate=False)

print("\nTip sample")
tip_raw_df.show(3, truncate=False)


### Phase 0–2 Summary

At this point, the notebook has established the full raw big-data layer for the integrated pipeline. The next stages will clean and structure the four source datasets separately, then combine them into enriched business-level and interaction-level outputs that support both Part A analytics and Part B recommendation.

## Phase 3. Source-Specific Cleaning and Structuring

### Phase 3 design updates based on raw-output inspection
- `checkin.json` stores timestamps as a single comma-separated `date` string, so it should be split and counted rather than treated like a ready-made array.
- `tip.json` includes `compliment_count`, which is worth preserving as a lightweight engagement signal.
- `business.json` remains the business catalog, but only a subset of attributes is stable enough to carry forward cleanly at this stage.
- `review.json` remains the main explicit interaction source and should be structured for both analytics and later recommendation input.


### Phase 3A. Business Cleaning

In [ ]:
business_clean_df = (
    business_raw_df
    .select(
        "business_id",
        F.col("name").alias("business_name"),
        F.trim(F.col("city")).alias("city_raw"),
        F.trim(F.col("state")).alias("state_raw"),
        F.col("postal_code"),
        F.col("stars").alias("business_stars"),
        F.col("review_count").alias("business_review_count"),
        F.col("is_open"),
        F.trim(F.col("categories")).alias("categories"),
        F.col("attributes.RestaurantsPriceRange2").alias("price_range_raw"),
        F.col("attributes.BusinessAcceptsCreditCards").alias("credit_card_flag_raw")
    )
    .filter(F.col("business_id").isNotNull())
    .withColumn(
        "city",
        F.when(F.col("city_raw").isNull() | (F.col("city_raw") == ""), F.lit("Unknown")).otherwise(F.col("city_raw"))
    )
    .withColumn(
        "state",
        F.when(F.col("state_raw").isNull() | (F.col("state_raw") == ""), F.lit("Unknown")).otherwise(F.col("state_raw"))
    )
    .withColumn(
        "category_list",
        F.when(
            F.col("categories").isNotNull(),
            F.split(F.regexp_replace(F.col("categories"), r"\s*,\s*", ","), ",")
        ).otherwise(F.expr("cast(array() as array<string>)"))
    )
    .withColumn(
        "price_range",
        F.expr("try_cast(nullif(trim(price_range_raw), 'None') as int)")
    )
    .withColumn(
        "credit_card_flag",
        F.when(F.lower(F.trim(F.col("credit_card_flag_raw"))) == "true", F.lit("Yes"))
         .when(F.lower(F.trim(F.col("credit_card_flag_raw"))) == "false", F.lit("No"))
         .otherwise(F.lit("Unknown"))
    )
    .drop("city_raw", "state_raw")
)

business_clean_df.printSchema()
business_clean_df.show(5, truncate=False)


### Phase 3B. Review Cleaning

In [ ]:
review_feature_df = (
    review_raw_df
    .select(
        "review_id",
        "user_id",
        "business_id",
        F.col("stars").alias("review_stars"),
        F.col("date").alias("review_date_raw"),
        F.col("text").alias("review_text_raw"),
        "useful",
        "funny",
        "cool"
    )
    .filter(F.col("business_id").isNotNull())
    .withColumn("review_text", F.coalesce(F.col("review_text_raw"), F.lit("")))
    .withColumn("review_timestamp", F.to_timestamp("review_date_raw"))
    .withColumn("review_date", F.to_date("review_timestamp"))
    .withColumn("review_year", F.year("review_date"))
    .withColumn("review_month", F.month("review_date"))
    .withColumn("review_length", F.length("review_text"))
    .withColumn("engagement_score", F.col("useful") + F.col("funny") + F.col("cool"))
    .drop("review_text_raw")
)

review_feature_df.printSchema()
review_feature_df.show(5, truncate=False)


### Phase 3C. Check-in Cleaning

In [ ]:
checkin_feature_df = (
    checkin_raw_df
    .select(
        "business_id",
        F.col("date").alias("checkin_dates_raw")
    )
    .filter(F.col("business_id").isNotNull())
    .withColumn(
        "checkin_timestamp_strings",
        F.when(
            F.col("checkin_dates_raw").isNotNull() & (F.trim(F.col("checkin_dates_raw")) != ""),
            F.split(F.col("checkin_dates_raw"), ",\\s*")
        ).otherwise(F.expr("cast(array() as array<string>)"))
    )
    .withColumn("total_checkins", F.size("checkin_timestamp_strings"))
    .withColumn(
        "first_checkin_timestamp",
        F.to_timestamp(F.element_at(F.array_sort(F.col("checkin_timestamp_strings")), 1))
    )
    .withColumn(
        "last_checkin_timestamp",
        F.to_timestamp(F.element_at(F.reverse(F.array_sort(F.col("checkin_timestamp_strings"))), 1))
    )
    .withColumn("first_checkin_year", F.year("first_checkin_timestamp"))
    .withColumn("last_checkin_year", F.year("last_checkin_timestamp"))
    .withColumn(
        "checkin_activity_bucket",
        F.when(F.col("total_checkins") >= 500, F.lit("Very High"))
         .when(F.col("total_checkins") >= 100, F.lit("High"))
         .when(F.col("total_checkins") >= 25, F.lit("Medium"))
         .when(F.col("total_checkins") > 0, F.lit("Low"))
         .otherwise(F.lit("None"))
    )
)

checkin_feature_df.printSchema()
checkin_feature_df.show(5, truncate=False)


### Phase 3D. Tip Cleaning

In [ ]:
tip_feature_df = (
    tip_raw_df
    .select(
        "user_id",
        "business_id",
        F.col("text").alias("tip_text_raw"),
        F.col("date").alias("tip_date_raw"),
        "compliment_count"
    )
    .filter(F.col("business_id").isNotNull())
    .withColumn("tip_text", F.coalesce(F.col("tip_text_raw"), F.lit("")))
    .withColumn("tip_timestamp", F.to_timestamp("tip_date_raw"))
    .withColumn("tip_date", F.to_date("tip_timestamp"))
    .withColumn("tip_year", F.year("tip_date"))
    .withColumn("tip_month", F.month("tip_date"))
    .withColumn("tip_length", F.length("tip_text"))
    .drop("tip_text_raw")
)

tip_feature_df.printSchema()
tip_feature_df.show(5, truncate=False)


### Phase 3A–3D Summary

The four raw Yelp sources have now been converted into structured layers for the rest of the integrated pipeline:

- `business_clean_df` provides business metadata and cleaned item features.
- `review_feature_df` provides explicit user-business interaction and engagement features.
- `checkin_feature_df` provides business-level activity signals.
- `tip_feature_df` provides lightweight interaction and supplementary engagement signals.

The next phases will use these structured layers for data quality diagnostics, business-level feature engineering, enriched business behavior construction, and later Part A analytics plus Part B recommendation.

## Phase 4. Data Quality Diagnostics

This phase evaluates the structured source layers before multi-source business enrichment. The focus is on coverage, nulls, duplicates, and business overlap across the four datasets.

In [ ]:
business_row_count = business_clean_df.count()
review_row_count = review_feature_df.count()
checkin_row_count = checkin_feature_df.count()
tip_row_count = tip_feature_df.count()

quality_overview_df = pd.DataFrame([
    {"dataset": "business_clean", "rows": business_row_count, "distinct_businesses": business_clean_df.select("business_id").distinct().count()},
    {"dataset": "review_feature", "rows": review_row_count, "distinct_businesses": review_feature_df.select("business_id").distinct().count()},
    {"dataset": "checkin_feature", "rows": checkin_row_count, "distinct_businesses": checkin_feature_df.select("business_id").distinct().count()},
    {"dataset": "tip_feature", "rows": tip_row_count, "distinct_businesses": tip_feature_df.select("business_id").distinct().count()},
])

quality_overview_df


In [ ]:
business_quality_diagnostics_df = business_clean_df.agg(
    F.count("*").alias("total_business_rows"),
    F.sum(F.when(F.col("business_id").isNull(), 1).otherwise(0)).alias("null_business_id"),
    F.sum(F.when(F.col("categories").isNull() | (F.trim(F.col("categories")) == ""), 1).otherwise(0)).alias("missing_categories"),
    F.sum(F.when(F.col("price_range").isNull(), 1).otherwise(0)).alias("missing_price_range"),
    F.sum(F.when(F.col("credit_card_flag") == "Unknown", 1).otherwise(0)).alias("unknown_credit_card_flag"),
    F.sum(F.when(F.col("city") == "Unknown", 1).otherwise(0)).alias("unknown_city"),
    F.sum(F.when(F.col("state") == "Unknown", 1).otherwise(0)).alias("unknown_state")
).toPandas()

business_quality_diagnostics_df["missing_categories_pct"] = round((business_quality_diagnostics_df["missing_categories"] / business_quality_diagnostics_df["total_business_rows"]) * 100, 2)
business_quality_diagnostics_df["missing_price_range_pct"] = round((business_quality_diagnostics_df["missing_price_range"] / business_quality_diagnostics_df["total_business_rows"]) * 100, 2)
business_quality_diagnostics_df["unknown_credit_card_flag_pct"] = round((business_quality_diagnostics_df["unknown_credit_card_flag"] / business_quality_diagnostics_df["total_business_rows"]) * 100, 2)
business_quality_diagnostics_df["unknown_city_pct"] = round((business_quality_diagnostics_df["unknown_city"] / business_quality_diagnostics_df["total_business_rows"]) * 100, 2)
business_quality_diagnostics_df["unknown_state_pct"] = round((business_quality_diagnostics_df["unknown_state"] / business_quality_diagnostics_df["total_business_rows"]) * 100, 2)

business_quality_diagnostics_df


In [ ]:
review_quality_diagnostics_df = review_feature_df.agg(
    F.count("*").alias("total_review_rows"),
    F.sum(F.when(F.col("review_id").isNull(), 1).otherwise(0)).alias("null_review_id"),
    F.sum(F.when(F.col("user_id").isNull(), 1).otherwise(0)).alias("null_user_id"),
    F.sum(F.when(F.col("business_id").isNull(), 1).otherwise(0)).alias("null_business_id"),
    F.sum(F.when(F.col("review_date").isNull(), 1).otherwise(0)).alias("null_review_date"),
    F.sum(F.when(F.col("review_text") == "", 1).otherwise(0)).alias("empty_review_text"),
    F.min("review_date").alias("earliest_review_date"),
    F.max("review_date").alias("latest_review_date")
).toPandas()

review_quality_diagnostics_df["null_review_date_pct"] = round((review_quality_diagnostics_df["null_review_date"] / review_quality_diagnostics_df["total_review_rows"]) * 100, 4)
review_quality_diagnostics_df["empty_review_text_pct"] = round((review_quality_diagnostics_df["empty_review_text"] / review_quality_diagnostics_df["total_review_rows"]) * 100, 4)

review_quality_diagnostics_df


In [ ]:
checkin_quality_diagnostics_df = checkin_feature_df.agg(
    F.count("*").alias("total_checkin_rows"),
    F.sum(F.when(F.col("business_id").isNull(), 1).otherwise(0)).alias("null_business_id"),
    F.sum(F.when(F.col("total_checkins") == 0, 1).otherwise(0)).alias("zero_checkin_businesses"),
    F.min("first_checkin_timestamp").alias("earliest_checkin_timestamp"),
    F.max("last_checkin_timestamp").alias("latest_checkin_timestamp")
).toPandas()

checkin_quality_diagnostics_df["zero_checkin_businesses_pct"] = round((checkin_quality_diagnostics_df["zero_checkin_businesses"] / checkin_quality_diagnostics_df["total_checkin_rows"]) * 100, 2)

checkin_quality_diagnostics_df


In [ ]:
tip_quality_diagnostics_df = tip_feature_df.agg(
    F.count("*").alias("total_tip_rows"),
    F.sum(F.when(F.col("user_id").isNull(), 1).otherwise(0)).alias("null_user_id"),
    F.sum(F.when(F.col("business_id").isNull(), 1).otherwise(0)).alias("null_business_id"),
    F.sum(F.when(F.col("tip_date").isNull(), 1).otherwise(0)).alias("null_tip_date"),
    F.sum(F.when(F.col("tip_text") == "", 1).otherwise(0)).alias("empty_tip_text"),
    F.min("tip_date").alias("earliest_tip_date"),
    F.max("tip_date").alias("latest_tip_date")
).toPandas()

tip_quality_diagnostics_df["null_tip_date_pct"] = round((tip_quality_diagnostics_df["null_tip_date"] / tip_quality_diagnostics_df["total_tip_rows"]) * 100, 4)
tip_quality_diagnostics_df["empty_tip_text_pct"] = round((tip_quality_diagnostics_df["empty_tip_text"] / tip_quality_diagnostics_df["total_tip_rows"]) * 100, 4)

tip_quality_diagnostics_df


In [ ]:
business_duplicate_count = business_row_count - business_clean_df.select("business_id").distinct().count()
review_duplicate_count = review_row_count - review_feature_df.select("review_id").distinct().count()
checkin_duplicate_count = checkin_row_count - checkin_feature_df.select("business_id").distinct().count()
tip_row_level_duplicates = tip_feature_df.groupBy("user_id", "business_id", "tip_date", "tip_text").count().filter(F.col("count") > 1).count()

duplicate_summary_df = pd.DataFrame([
    {"dataset": "business_clean", "duplicate_key_rows": business_duplicate_count, "key_used": "business_id"},
    {"dataset": "review_feature", "duplicate_key_rows": review_duplicate_count, "key_used": "review_id"},
    {"dataset": "checkin_feature", "duplicate_key_rows": checkin_duplicate_count, "key_used": "business_id"},
    {"dataset": "tip_feature", "duplicate_key_rows": tip_row_level_duplicates, "key_used": "user_id + business_id + tip_date + tip_text"},
])

duplicate_summary_df


In [ ]:
business_ids_df = business_clean_df.select("business_id").distinct()
review_business_ids_df = review_feature_df.select("business_id").distinct()
checkin_business_ids_df = checkin_feature_df.select("business_id").distinct()
tip_business_ids_df = tip_feature_df.select("business_id").distinct()

business_base_count = business_ids_df.count()

overlap_summary_df = pd.DataFrame([
    {
        "comparison": "businesses also in review data",
        "matched_businesses": business_ids_df.join(review_business_ids_df, on="business_id", how="inner").count(),
        "base_businesses": business_base_count,
    },
    {
        "comparison": "businesses also in checkin data",
        "matched_businesses": business_ids_df.join(checkin_business_ids_df, on="business_id", how="inner").count(),
        "base_businesses": business_base_count,
    },
    {
        "comparison": "businesses also in tip data",
        "matched_businesses": business_ids_df.join(tip_business_ids_df, on="business_id", how="inner").count(),
        "base_businesses": business_base_count,
    },
    {
        "comparison": "businesses present in all four layers",
        "matched_businesses": business_ids_df.join(review_business_ids_df, on="business_id", how="inner").join(checkin_business_ids_df, on="business_id", how="inner").join(tip_business_ids_df, on="business_id", how="inner").count(),
        "base_businesses": business_base_count,
    },
])

overlap_summary_df["coverage_pct"] = round((overlap_summary_df["matched_businesses"] / overlap_summary_df["base_businesses"]) * 100, 2)
overlap_summary_df


### Phase 4 Summary

The diagnostics in this phase establish whether the cleaned source layers are robust enough for multi-source feature engineering. The next stage will aggregate review, check-in, and tip behavior to the business level, so understanding source coverage and overlap now is essential for building a defensible enriched business layer.

## Phase 5. Feature Engineering at Business Level

The earlier phases established that `review_feature_df` provides the complete interaction backbone, `checkin_feature_df` provides strong but not universal activity coverage, and `tip_feature_df` provides useful supplementary engagement with partial coverage. This phase turns each source into business-level engineered feature layers that can later be joined into a single enriched business behavior dataset.

### Phase 5A. Business Feature Layer

In [ ]:
business_feature_df = (
    business_clean_df
    .withColumn("category_count", F.size("category_list"))
    .withColumn("has_categories", F.col("category_count") > 0)
    .withColumn(
        "price_range_label",
        F.when(F.col("price_range") == 1, F.lit("Low"))
         .when(F.col("price_range") == 2, F.lit("Medium"))
         .when(F.col("price_range") == 3, F.lit("High"))
         .when(F.col("price_range") == 4, F.lit("Premium"))
         .otherwise(F.lit("Unknown"))
    )
    .withColumn(
        "business_review_count_bucket",
        F.when(F.col("business_review_count") >= 500, F.lit("Very High"))
         .when(F.col("business_review_count") >= 100, F.lit("High"))
         .when(F.col("business_review_count") >= 25, F.lit("Medium"))
         .when(F.col("business_review_count") > 0, F.lit("Low"))
         .otherwise(F.lit("None"))
    )
    .withColumn("location_key", F.concat_ws(", ", F.col("city"), F.col("state")))
)

business_feature_df.select(
    "business_id",
    "business_name",
    "city",
    "state",
    "business_stars",
    "business_review_count",
    "category_count",
    "has_categories",
    "price_range",
    "price_range_label",
    "credit_card_flag",
    "business_review_count_bucket"
).show(5, truncate=False)


### Phase 5B. Review-Derived Business Aggregates

In [ ]:
review_business_agg_df = (
    review_feature_df
    .groupBy("business_id")
    .agg(
        F.count("review_id").alias("actual_review_count"),
        F.avg("review_stars").alias("avg_review_stars"),
        F.avg("engagement_score").alias("avg_engagement_per_review"),
        F.sum("engagement_score").alias("total_review_engagement"),
        F.avg("review_length").alias("avg_review_length"),
        F.min("review_year").alias("first_review_year"),
        F.max("review_year").alias("last_review_year")
    )
    .withColumn(
        "review_active_years",
        F.when(
            F.col("first_review_year").isNotNull() & F.col("last_review_year").isNotNull(),
            F.col("last_review_year") - F.col("first_review_year") + F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "actual_review_count_bucket",
        F.when(F.col("actual_review_count") >= 500, F.lit("Very High"))
         .when(F.col("actual_review_count") >= 100, F.lit("High"))
         .when(F.col("actual_review_count") >= 25, F.lit("Medium"))
         .when(F.col("actual_review_count") > 0, F.lit("Low"))
         .otherwise(F.lit("None"))
    )
)

review_business_agg_df.show(5, truncate=False)


### Phase 5C. Check-in Business Aggregates

In [ ]:
checkin_business_agg_df = checkin_feature_df.select(
    "business_id",
    "total_checkins",
    "first_checkin_year",
    "last_checkin_year",
    "checkin_activity_bucket"
)

checkin_business_agg_df.show(5, truncate=False)


### Phase 5D. Tip-Derived Business Aggregates

In [ ]:
tip_business_agg_df = (
    tip_feature_df
    .groupBy("business_id")
    .agg(
        F.count("*").alias("tip_count"),
        F.avg("tip_length").alias("avg_tip_length"),
        F.sum("compliment_count").alias("total_tip_compliments"),
        F.avg("compliment_count").alias("avg_tip_compliments"),
        F.min("tip_year").alias("first_tip_year"),
        F.max("tip_year").alias("last_tip_year")
    )
    .withColumn(
        "tip_active_years",
        F.when(
            F.col("first_tip_year").isNotNull() & F.col("last_tip_year").isNotNull(),
            F.col("last_tip_year") - F.col("first_tip_year") + F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "tip_count_bucket",
        F.when(F.col("tip_count") >= 500, F.lit("Very High"))
         .when(F.col("tip_count") >= 100, F.lit("High"))
         .when(F.col("tip_count") >= 25, F.lit("Medium"))
         .when(F.col("tip_count") > 0, F.lit("Low"))
         .otherwise(F.lit("None"))
    )
)

tip_business_agg_df.show(5, truncate=False)


### Phase 5 Summary

This phase converts the structured source layers into business-level engineered feature tables:

- `business_feature_df` captures business metadata and structural features.
- `review_business_agg_df` captures review-driven quality and engagement history.
- `checkin_business_agg_df` captures activity intensity from check-ins.
- `tip_business_agg_df` captures lightweight interaction and supplementary engagement from tips.

The next phase will join these business-level layers into a single enriched business behavior dataset that becomes the main analytical asset of Part A and the main feature source for Part B.

## Phase 6. Build the Enriched Business Behavior Layer

This phase joins the business catalog with review-derived quality and engagement features, check-in activity features, and tip-derived supplementary engagement features. Review coverage is treated as the dense backbone, while check-in and tip layers are treated as enrichments with safe defaults for missing businesses.

In [ ]:
business_behavior_enriched_df = (
    business_feature_df
    .join(review_business_agg_df, on="business_id", how="left")
    .join(checkin_business_agg_df, on="business_id", how="left")
    .join(tip_business_agg_df, on="business_id", how="left")
    .withColumn("actual_review_count", F.coalesce(F.col("actual_review_count"), F.lit(0)))
    .withColumn("avg_review_stars", F.coalesce(F.col("avg_review_stars"), F.col("business_stars")))
    .withColumn("avg_engagement_per_review", F.coalesce(F.col("avg_engagement_per_review"), F.lit(0.0)))
    .withColumn("total_review_engagement", F.coalesce(F.col("total_review_engagement"), F.lit(0)))
    .withColumn("avg_review_length", F.coalesce(F.col("avg_review_length"), F.lit(0.0)))
    .withColumn("review_active_years", F.coalesce(F.col("review_active_years"), F.lit(0)))
    .withColumn("actual_review_count_bucket", F.coalesce(F.col("actual_review_count_bucket"), F.lit("None")))
    .withColumn("total_checkins", F.coalesce(F.col("total_checkins"), F.lit(0)))
    .withColumn("checkin_activity_bucket", F.coalesce(F.col("checkin_activity_bucket"), F.lit("None")))
    .withColumn("tip_count", F.coalesce(F.col("tip_count"), F.lit(0)))
    .withColumn("avg_tip_length", F.coalesce(F.col("avg_tip_length"), F.lit(0.0)))
    .withColumn("total_tip_compliments", F.coalesce(F.col("total_tip_compliments"), F.lit(0)))
    .withColumn("avg_tip_compliments", F.coalesce(F.col("avg_tip_compliments"), F.lit(0.0)))
    .withColumn("tip_active_years", F.coalesce(F.col("tip_active_years"), F.lit(0)))
    .withColumn("tip_count_bucket", F.coalesce(F.col("tip_count_bucket"), F.lit("None")))
)

business_behavior_enriched_df.printSchema()


In [ ]:
business_behavior_enriched_df.select(
    "business_id",
    "business_name",
    "city",
    "state",
    "business_stars",
    "avg_review_stars",
    "business_review_count",
    "actual_review_count",
    "avg_engagement_per_review",
    "total_checkins",
    "tip_count",
    "price_range_label",
    "category_count",
    "business_review_count_bucket",
    "actual_review_count_bucket",
    "checkin_activity_bucket",
    "tip_count_bucket"
).show(10, truncate=False)


In [ ]:
business_behavior_layer_summary_df = business_behavior_enriched_df.agg(
    F.count("*").alias("total_businesses"),
    F.sum(F.when(F.col("actual_review_count") > 0, 1).otherwise(0)).alias("businesses_with_reviews"),
    F.sum(F.when(F.col("total_checkins") > 0, 1).otherwise(0)).alias("businesses_with_checkins"),
    F.sum(F.when(F.col("tip_count") > 0, 1).otherwise(0)).alias("businesses_with_tips"),
    F.avg("actual_review_count").alias("avg_actual_review_count"),
    F.avg("total_checkins").alias("avg_total_checkins"),
    F.avg("tip_count").alias("avg_tip_count")
)

business_behavior_layer_summary_df.show(truncate=False)


### Phase 6 Summary

The notebook now has its main enriched business behavior layer: `business_behavior_enriched_df`. This dataset combines business metadata, review-derived quality and engagement, check-in activity, and tip-derived supplementary behavior in one business-level representation. It becomes the central analytical asset for Part A and the primary feature source for Part B recommendation.

## Phase 7. Create the Recommendation-Ready Interaction Layer

The project’s main recommendation method will be content-based, but the pipeline should still expose a compact interaction layer because it preserves the user-business relationship established in the raw review data. This makes the Part A to Part B pipeline more complete and keeps the recommendation side grounded in structured interaction evidence.

In [ ]:
user_business_interactions_df = review_feature_df.select(
    "user_id",
    "business_id",
    F.col("review_stars").alias("rating"),
    "review_date",
    "review_year",
    "review_month",
    "engagement_score"
)

user_business_interactions_df.printSchema()
user_business_interactions_df.show(10, truncate=False)


In [ ]:
interaction_layer_summary_df = user_business_interactions_df.agg(
    F.count("*").alias("total_interactions"),
    F.countDistinct("user_id").alias("distinct_users"),
    F.countDistinct("business_id").alias("distinct_businesses"),
    F.avg("rating").alias("avg_rating"),
    F.avg("engagement_score").alias("avg_engagement_score")
)

interaction_layer_summary_df.show(truncate=False)


In [ ]:
user_interaction_profile_df = user_business_interactions_df.groupBy("user_id").agg(
    F.count("business_id").alias("interaction_count"),
    F.avg("rating").alias("avg_user_rating"),
    F.avg("engagement_score").alias("avg_user_engagement")
)

user_interaction_profile_df.show(5, truncate=False)


### Phase 7 Summary

The notebook now has a compact recommendation-ready interaction layer: `user_business_interactions_df`. This keeps the user-business relationship from reviews in a structured form without repeating the full review dataset, and it strengthens the Part A to Part B transition by exposing both a business-level feature layer and an interaction-level layer.

## Phase 8. Analysis of Business Quality

The main question in this phase is: **what characteristics are associated with highly rated businesses?** The enriched business layer is used as the primary analytical source because it combines business metadata with review-derived quality signals.

In [ ]:
business_categories_exploded_df = business_behavior_enriched_df.withColumn(
    "category",
    F.explode_outer("category_list")
)

business_categories_exploded_df = business_categories_exploded_df.filter(
    F.col("category").isNotNull() & (F.trim(F.col("category")) != "")
)


In [ ]:
business_rating_distribution_df = business_behavior_enriched_df.groupBy("business_stars").count().orderBy("business_stars")
business_rating_distribution_df.show(truncate=False)


In [ ]:
quality_by_category_df = (
    business_categories_exploded_df
    .groupBy("category")
    .agg(
        F.countDistinct("business_id").alias("business_count"),
        F.avg("business_stars").alias("avg_business_stars"),
        F.avg("avg_review_stars").alias("avg_review_stars")
    )
    .filter(F.col("business_count") >= 100)
    .orderBy(F.desc("avg_review_stars"), F.desc("business_count"))
)
quality_by_category_df.show(15, truncate=False)


In [ ]:
quality_by_state_df = (
    business_behavior_enriched_df
    .groupBy("state")
    .agg(
        F.count("business_id").alias("business_count"),
        F.avg("avg_review_stars").alias("avg_review_stars")
    )
    .filter(F.col("business_count") >= 100)
    .orderBy(F.desc("business_count"))
)
quality_by_state_df.show(15, truncate=False)


In [ ]:
quality_by_price_range_df = (
    business_behavior_enriched_df
    .groupBy("price_range_label")
    .agg(
        F.count("business_id").alias("business_count"),
        F.avg("avg_review_stars").alias("avg_review_stars"),
        F.avg("actual_review_count").alias("avg_actual_review_count")
    )
    .orderBy(F.desc("business_count"))
)
quality_by_price_range_df.show(truncate=False)


## Phase 9. Analysis of Engagement

The main question in this phase is: **what characteristics are associated with highly engaged businesses?** Engagement is interpreted through review-level reactions and tip-related signals that were aggregated to the business level.

In [ ]:
engagement_by_category_df = (
    business_categories_exploded_df
    .groupBy("category")
    .agg(
        F.countDistinct("business_id").alias("business_count"),
        F.avg("avg_engagement_per_review").alias("avg_engagement_per_review"),
        F.avg("tip_count").alias("avg_tip_count"),
        F.avg("avg_tip_compliments").alias("avg_tip_compliments")
    )
    .filter(F.col("business_count") >= 100)
    .orderBy(F.desc("avg_engagement_per_review"), F.desc("avg_tip_count"))
)
engagement_by_category_df.show(15, truncate=False)


In [ ]:
engagement_by_price_range_df = (
    business_behavior_enriched_df
    .groupBy("price_range_label")
    .agg(
        F.count("business_id").alias("business_count"),
        F.avg("avg_engagement_per_review").alias("avg_engagement_per_review"),
        F.avg("tip_count").alias("avg_tip_count")
    )
    .orderBy(F.desc("business_count"))
)
engagement_by_price_range_df.show(truncate=False)


In [ ]:
top_businesses_by_engagement_df = (
    business_behavior_enriched_df
    .select(
        "business_id",
        "business_name",
        "city",
        "state",
        "avg_review_stars",
        "actual_review_count",
        "avg_engagement_per_review",
        "tip_count"
    )
    .filter(F.col("actual_review_count") >= 25)
    .orderBy(F.desc("avg_engagement_per_review"), F.desc("actual_review_count"))
)
top_businesses_by_engagement_df.show(15, truncate=False)


## Phase 10. Analysis of Activity

The main question in this phase is: **what characteristics are associated with highly active businesses?** Activity is measured through actual review volume, check-in counts, tip counts, and temporal review behavior.

In [ ]:
activity_by_category_df = (
    business_categories_exploded_df
    .groupBy("category")
    .agg(
        F.countDistinct("business_id").alias("business_count"),
        F.avg("actual_review_count").alias("avg_actual_review_count"),
        F.avg("total_checkins").alias("avg_total_checkins"),
        F.avg("tip_count").alias("avg_tip_count")
    )
    .filter(F.col("business_count") >= 100)
    .orderBy(F.desc("avg_total_checkins"), F.desc("avg_actual_review_count"))
)
activity_by_category_df.show(15, truncate=False)


In [ ]:
activity_by_state_df = (
    business_behavior_enriched_df
    .groupBy("state")
    .agg(
        F.count("business_id").alias("business_count"),
        F.avg("actual_review_count").alias("avg_actual_review_count"),
        F.avg("total_checkins").alias("avg_total_checkins")
    )
    .filter(F.col("business_count") >= 100)
    .orderBy(F.desc("avg_total_checkins"), F.desc("avg_actual_review_count"))
)
activity_by_state_df.show(15, truncate=False)


In [ ]:
review_activity_over_time_df = (
    review_feature_df
    .groupBy("review_year")
    .agg(
        F.count("review_id").alias("review_count"),
        F.avg("engagement_score").alias("avg_engagement_score")
    )
    .orderBy("review_year")
)
review_activity_over_time_df.show(truncate=False)


## Phase 11. Integrated Analytics: Quality, Engagement, and Activity Together

This phase brings the three dimensions together and asks: **which business characteristics best balance quality, engagement, and activity?** This is the strongest analytical bridge from Part A to Part B because it reveals the enriched business profiles that later support recommendation.

In [ ]:
balanced_category_performance_df = (
    business_categories_exploded_df
    .groupBy("category")
    .agg(
        F.countDistinct("business_id").alias("business_count"),
        F.avg("avg_review_stars").alias("avg_review_stars"),
        F.avg("avg_engagement_per_review").alias("avg_engagement_per_review"),
        F.avg("total_checkins").alias("avg_total_checkins"),
        F.avg("tip_count").alias("avg_tip_count")
    )
    .filter(F.col("business_count") >= 100)
    .withColumn(
        "balanced_score",
        (F.col("avg_review_stars") * F.lit(0.45))
        + (F.col("avg_engagement_per_review") * F.lit(0.20))
        + (F.log10(F.col("avg_total_checkins") + F.lit(1)) * F.lit(0.20))
        + (F.log10(F.col("avg_tip_count") + F.lit(1)) * F.lit(0.15))
    )
    .orderBy(F.desc("balanced_score"), F.desc("business_count"))
)
balanced_category_performance_df.show(20, truncate=False)


In [ ]:
quality_activity_businesses_df = (
    business_behavior_enriched_df
    .filter(F.col("actual_review_count") >= 50)
    .withColumn(
        "business_balance_score",
        (F.col("avg_review_stars") * F.lit(0.5))
        + (F.log10(F.col("total_checkins") + F.lit(1)) * F.lit(0.25))
        + (F.log10(F.col("tip_count") + F.lit(1)) * F.lit(0.10))
        + (F.col("avg_engagement_per_review") * F.lit(0.15))
    )
    .select(
        "business_id",
        "business_name",
        "city",
        "state",
        "avg_review_stars",
        "actual_review_count",
        "avg_engagement_per_review",
        "total_checkins",
        "tip_count",
        "business_balance_score"
    )
    .orderBy(F.desc("business_balance_score"), F.desc("actual_review_count"))
)
quality_activity_businesses_df.show(20, truncate=False)


### Phase 8–11 Summary

These four analytical phases turn the enriched business layer into a direct answer to the first half of the project problem. The notebook now identifies how ratings, engagement, and activity vary across categories, price levels, and locations, and it also surfaces more balanced business and category profiles that are especially valuable for the recommendation stage.

## Phase 12. Visualizations for Part A

This phase turns the strongest analytical outputs into a focused set of visuals for the report and slides. The figures are built from Spark-aggregated outputs and then converted into small plotting tables only at the final display stage.

In [ ]:
sns.set_theme(style="whitegrid")

def save_bar_chart(df_pd, x_col, y_col, title, output_name, horizontal=False, figsize=(10, 6), color="#2A6F97"):
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(data=df_pd, x=x_col, y=y_col, color=color, ax=ax)
    if not horizontal:
        ax.tick_params(axis="x", rotation=45)
        for label in ax.get_xticklabels():
            label.set_ha("right")
    ax.set_title(title)
    fig.tight_layout()
    save_path = FIGURES_DIR / output_name
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    print(f"Saved chart to: {save_path}")
    plt.show()
    plt.close(fig)
    return save_path


In [ ]:
top_states_business_count_df = business_behavior_enriched_df.groupBy("state").count().orderBy(F.desc("count")).limit(10)
top_states_business_count_pd = top_states_business_count_df.toPandas()
save_bar_chart(top_states_business_count_pd, "state", "count", "Top States by Yelp Business Count", "part_a_top_states_business_count.png")


In [ ]:
top_categories_business_count_df = business_categories_exploded_df.groupBy("category").count().orderBy(F.desc("count")).limit(10)
top_categories_business_count_pd = top_categories_business_count_df.toPandas()
save_bar_chart(top_categories_business_count_pd, "count", "category", "Top Categories by Business Count", "part_a_top_categories_business_count.png", horizontal=True, figsize=(10, 7))


In [ ]:
business_rating_distribution_pd = business_rating_distribution_df.toPandas()
save_bar_chart(business_rating_distribution_pd, "business_stars", "count", "Business Rating Distribution", "part_a_business_rating_distribution.png")


In [ ]:
top_quality_categories_pd = quality_by_category_df.limit(10).toPandas()
save_bar_chart(top_quality_categories_pd, "avg_review_stars", "category", "Average Review Stars by Top Quality Categories", "part_a_avg_review_stars_by_category.png", horizontal=True, figsize=(10, 7))


In [ ]:
top_engagement_categories_pd = engagement_by_category_df.limit(10).toPandas()
save_bar_chart(top_engagement_categories_pd, "avg_engagement_per_review", "category", "Average Engagement per Review by Category", "part_a_avg_engagement_by_category.png", horizontal=True, figsize=(10, 7))


In [ ]:
review_activity_over_time_pd = review_activity_over_time_df.toPandas()
plt.figure(figsize=(10, 6))
sns.lineplot(data=review_activity_over_time_pd, x="review_year", y="review_count", marker="o", color="#2A6F97")
plt.title("Review Activity Over Time")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "part_a_review_activity_over_time.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
top_activity_categories_pd = activity_by_category_df.limit(10).toPandas()
save_bar_chart(top_activity_categories_pd, "avg_total_checkins", "category", "Average Check-in Activity by Category", "part_a_avg_checkins_by_category.png", horizontal=True, figsize=(10, 7))


In [ ]:
balanced_category_performance_pd = balanced_category_performance_df.limit(10).toPandas()
save_bar_chart(balanced_category_performance_pd, "balanced_score", "category", "Balanced Category Performance", "part_a_balanced_category_performance.png", horizontal=True, figsize=(10, 7))


## Phase 13. Part A Findings and Interpretation

The results from Phases 8–12 help answer the first half of my project question. In simple terms, they show that highly rated, highly engaged, and highly active businesses do have recognisable patterns, and those patterns can be studied using the enriched Yelp data I prepared earlier.


### Key Findings

1. **Business quality is not evenly distributed across categories and locations.**  
The quality analysis shows that average review-based business ratings vary meaningfully across categories, states, and price levels. This suggests that stronger perceived business quality is associated with identifiable business characteristics rather than being randomly distributed.

2. **Engagement patterns differ from rating patterns.**  
Some categories and businesses receive much stronger review engagement than others, even when they are not always the highest-rated. The results also indicate a long-tail effect where a small number of businesses have extremely high average engagement per review, so engagement should be interpreted as a useful but potentially skewed signal rather than a direct substitute for quality.

3. **Business activity is broader than review volume alone.**  
Check-ins and tips add a second view of business activity. The activity analysis shows that highly active businesses are not defined only by review counts, which justifies using multiple behavioral signals in the enriched business layer. At the same time, check-in and tip coverage are not universal, so they should be treated as enrichment signals rather than mandatory features for every business.

4. **Balanced business and category profiles are more useful than single-metric rankings.**  
The integrated analytics phase shows that more informative business and category profiles combine quality, engagement, and activity together. However, some of the top balanced categories are niche or place-based categories such as landmarks, hiking, or beaches, which means category-level ranking should be interpreted carefully and later filtered to user-relevant recommendation contexts.

5. **The enriched business behavior layer is a strong bridge to recommendation.**  
By combining business metadata with review, check-in, and tip features, the project produces a structured business representation that is directly useful for Part B recommendation. This makes the transition from analytics to recommendation explicit and defensible.


### Interpretation for the Project Problem

The first half of the project problem asked how big data analytics can reveal the characteristics of highly rated, highly engaged, and highly active businesses. The results show that these characteristics emerge from a combination of category, location, price positioning, review behavior, check-in activity, and tip-based signals rather than from one metric alone. They also show that some signals, especially engagement, can be skewed by extreme cases and that some top-ranked categories are highly contextual. This justifies the Part B design, where recommendation is built on a richer structured business representation from Part A and where later recommendation logic should emphasize relevant business similarity rather than blindly following a single score.

## Phase 14. Export Processed Outputs for Part B

This phase creates the explicit handoff from Part A to Part B. Because the selected Part B design uses a popularity baseline and a main content-based recommender, the most important downstream assets are the business-level processed outputs. In this environment, those outputs are prepared for direct reuse in Part B through the CSV-based handoff below.

In [ ]:
# Earlier parquet export paths were retained only during development.
# The final integrated pipeline uses the CSV-based handoff below for Part B.


### CSV-Based Export Handoff

Because the selected Part B design uses a popularity baseline and a main content-based recommender, the most important downstream assets are the business-level processed outputs rather than the full multi-million-row interaction tables. The following handoff saves those business-level outputs as CSV for direct reuse in Part B while keeping the large interaction layers available in Spark memory inside the integrated pipeline.

In [ ]:
BUSINESS_FEATURE_CSV_PATH = PROCESSED_DATA_DIR / "business_feature.csv"
CHECKIN_FEATURE_CSV_PATH = PROCESSED_DATA_DIR / "checkin_feature.csv"
TIP_FEATURE_CSV_PATH = PROCESSED_DATA_DIR / "tip_feature.csv"
BUSINESS_BEHAVIOR_CSV_PATH = PROCESSED_DATA_DIR / "business_behavior_enriched.csv"


In [ ]:
business_feature_df.toPandas().to_csv(BUSINESS_FEATURE_CSV_PATH, index=False)
checkin_feature_df.toPandas().to_csv(CHECKIN_FEATURE_CSV_PATH, index=False)
tip_feature_df.toPandas().to_csv(TIP_FEATURE_CSV_PATH, index=False)
business_behavior_enriched_df.toPandas().to_csv(BUSINESS_BEHAVIOR_CSV_PATH, index=False)

print("Business-level processed CSV outputs saved successfully.")


In [ ]:
csv_processed_output_summary_df = pd.DataFrame([
    {"dataset": "business_feature", "path": str(BUSINESS_FEATURE_CSV_PATH), "role_for_part_b": "supporting business features"},
    {"dataset": "checkin_feature", "path": str(CHECKIN_FEATURE_CSV_PATH), "role_for_part_b": "supporting activity features"},
    {"dataset": "tip_feature", "path": str(TIP_FEATURE_CSV_PATH), "role_for_part_b": "supporting engagement features"},
    {"dataset": "business_behavior_enriched", "path": str(BUSINESS_BEHAVIOR_CSV_PATH), "role_for_part_b": "main business-level recommendation input"},
    {"dataset": "review_feature", "path": "kept in Spark memory", "role_for_part_b": "not required as a saved file for the chosen main method"},
    {"dataset": "user_business_interactions", "path": "kept in Spark memory", "role_for_part_b": "prepared interaction layer, not the main saved Part B input"},
])

csv_processed_output_summary_df


With this revised export strategy, Part B should load the saved CSV business-level outputs from `data/processed`, especially `business_behavior_enriched.csv`. The large interaction layers remain part of the integrated pipeline and can still be referenced in-memory if needed, but they do not need to be physically exported for the chosen Part B methods.

## Phase 15. Load Processed Outputs from Part A

This phase begins Part B by loading the processed outputs created in Part A. The recommendation stage now depends on structured business-level data rather than touching the raw Yelp JSON files again.

In [ ]:
part_b_business_source = "in_memory"
part_b_checkin_source = "in_memory"
part_b_tip_source = "in_memory"

if 'business_behavior_enriched_df' in globals():
    part_b_business_behavior_df = business_behavior_enriched_df
elif BUSINESS_BEHAVIOR_CSV_PATH.exists():
    part_b_business_behavior_df = spark.read.option("header", True).option("inferSchema", True).csv(str(BUSINESS_BEHAVIOR_CSV_PATH))
    part_b_business_source = "csv"
else:
    raise FileNotFoundError("No Part B business behavior source is available.")

if 'checkin_feature_df' in globals():
    part_b_checkin_feature_df = checkin_feature_df
elif CHECKIN_FEATURE_CSV_PATH.exists():
    part_b_checkin_feature_df = spark.read.option("header", True).option("inferSchema", True).csv(str(CHECKIN_FEATURE_CSV_PATH))
    part_b_checkin_source = "csv"
else:
    part_b_checkin_feature_df = None

if 'tip_feature_df' in globals():
    part_b_tip_feature_df = tip_feature_df
elif TIP_FEATURE_CSV_PATH.exists():
    part_b_tip_feature_df = spark.read.option("header", True).option("inferSchema", True).csv(str(TIP_FEATURE_CSV_PATH))
    part_b_tip_source = "csv"
else:
    part_b_tip_feature_df = None

category_dtype = dict(part_b_business_behavior_df.dtypes).get("category_list")
if category_dtype != "array<string>":
    part_b_business_behavior_df = (
        part_b_business_behavior_df
        .withColumn("category_list", F.coalesce(F.col("category_list").cast("string"), F.lit("")))
        .withColumn("category_list", F.regexp_replace(F.col("category_list"), r"^\[|\]$", ""))
        .withColumn("category_list", F.regexp_replace(F.col("category_list"), r"'", ""))
        .withColumn("category_list", F.regexp_replace(F.col("category_list"), r'"', ""))
        .withColumn("category_list", F.split(F.col("category_list"), r",\s*|\s{2,}"))
        .withColumn("category_list", F.expr("filter(category_list, x -> x is not null and trim(x) <> '')"))
    )

part_b_load_summary_df = pd.DataFrame([
    {"dataset": "business_behavior_enriched", "source": part_b_business_source},
    {"dataset": "checkin_feature", "source": part_b_checkin_source},
    {"dataset": "tip_feature", "source": part_b_tip_source},
])

part_b_load_summary_df


In [ ]:
part_b_business_behavior_df.select(
    "business_id", "business_name", "city", "state", "category_list",
    "avg_review_stars", "actual_review_count", "total_checkins", "tip_count"
).show(5, truncate=False)


## Phase 16. Define the Recommendation Problem for Part B

In Part B, my goal is to recommend relevant businesses using the enriched business information created in Part A. I use two recommendation views:

- a **popularity-based baseline** to show strong businesses overall
- a **content-based recommender** to show businesses that are similar to a selected business

The baseline is there as a simple benchmark. The main solution is the content-based recommender because it makes better use of the features I built earlier.


## Phase 17. Popularity-Based Recommendation Baseline

The baseline recommends businesses that combine strong average review quality with strong observed business activity. This provides a simple benchmark before moving to similarity-based recommendation.

In [ ]:
popularity_recommendation_base_df = (
    part_b_business_behavior_df
    .filter(F.col("is_open") == 1)
    .withColumn("popularity_score",
        (F.col("avg_review_stars") * F.lit(0.50)) +
        (F.log10(F.col("actual_review_count") + F.lit(1)) * F.lit(0.20)) +
        (F.log10(F.col("total_checkins") + F.lit(1)) * F.lit(0.20)) +
        (F.log10(F.col("tip_count") + F.lit(1)) * F.lit(0.10))
    )
)

popularity_recommendation_base_df.select(
    "business_name", "city", "state", "avg_review_stars", "actual_review_count", "total_checkins", "tip_count", "popularity_score"
).orderBy(F.desc("popularity_score"), F.desc("actual_review_count")).show(10, truncate=False)


In [ ]:
def recommend_popular_businesses(city=None, category=None, top_n=10, min_reviews=50):
    df = popularity_recommendation_base_df.filter(F.col("actual_review_count") >= min_reviews)
    if city is not None:
        df = df.filter(F.col("city") == city)
    if category is not None:
        df = df.filter(F.array_contains(F.col("category_list"), category))
    return df.select(
        "business_id", "business_name", "city", "state", "avg_review_stars",
        "actual_review_count", "total_checkins", "tip_count", "popularity_score"
    ).orderBy(F.desc("popularity_score"), F.desc("avg_review_stars"), F.desc("actual_review_count")).limit(top_n)

top_city_for_baseline = (
    popularity_recommendation_base_df
    .groupBy("city")
    .count()
    .orderBy(F.desc("count"))
    .first()["city"]
)

print(f"Top baseline city example: {top_city_for_baseline}")
recommend_popular_businesses(city=top_city_for_baseline, top_n=5).show(truncate=False)


## Phase 18. Prepare Content-Based Recommendation Features

The main recommender uses the enriched business representation from Part A. The feature layer combines business metadata, category signals, price positioning, review quality, and activity signals into a recommendation-ready business space.

In [ ]:
content_recommendation_feature_df = (
    part_b_business_behavior_df
    .filter(F.col("is_open") == 1)
    .withColumn("effective_stars", F.coalesce(F.col("avg_review_stars"), F.col("business_stars")))
    .withColumn("stars_band",
        F.when(F.col("effective_stars") >= 4.5, F.lit("very_high"))
         .when(F.col("effective_stars") >= 4.0, F.lit("high"))
         .when(F.col("effective_stars") >= 3.5, F.lit("medium"))
         .otherwise(F.lit("lower"))
    )
    .withColumn("activity_intensity_score",
        F.log10(F.col("actual_review_count") + F.lit(1)) +
        F.log10(F.col("total_checkins") + F.lit(1)) +
        F.log10(F.col("tip_count") + F.lit(1))
    )
)

content_recommendation_feature_df.select(
    "business_id", "business_name", "city", "state", "category_list", "price_range_label", "effective_stars", "activity_intensity_score"
).show(5, truncate=False)


## Phase 19. Build the Main Content-Based Recommender

The main recommender compares businesses by shared categories, location, price range, review quality, and activity profile. The scoring logic is deliberately transparent so that the resulting recommendations remain explainable.

In [ ]:
def recommend_similar_businesses(reference_business_id, top_n=10, same_state_only=False):
    reference_rows = (
        content_recommendation_feature_df
        .filter(F.col("business_id") == reference_business_id)
        .select(
            "business_id", "business_name", "city", "state", "category_list", "price_range",
            "price_range_label", "effective_stars", "activity_intensity_score"
        )
        .limit(1)
        .collect()
    )
    if not reference_rows:
        raise ValueError(f"Reference business_id not found: {reference_business_id}")

    ref = reference_rows[0].asDict()
    ref_categories = ref.get("category_list") or []
    category_expr = F.array(*[F.lit(cat) for cat in ref_categories]) if ref_categories else None

    candidates_df = content_recommendation_feature_df.filter(F.col("business_id") != reference_business_id)
    if same_state_only and ref.get("state") is not None:
        candidates_df = candidates_df.filter(F.col("state") == F.lit(ref["state"]))

    shared_category_score = (
        F.size(F.array_intersect(F.col("category_list"), category_expr)).cast("double") * F.lit(3.0)
        if category_expr is not None else F.lit(0.0)
    )
    city_score = F.when(F.col("city") == F.lit(ref.get("city")), F.lit(2.0)).otherwise(F.lit(0.0))
    state_score = F.when(F.col("state") == F.lit(ref.get("state")), F.lit(1.0)).otherwise(F.lit(0.0))
    price_score = F.when(F.col("price_range") == F.lit(ref.get("price_range")), F.lit(1.25)).otherwise(F.lit(0.0))
    stars_score = F.greatest(F.lit(0.0), F.lit(1.5) - F.abs(F.col("effective_stars") - F.lit(float(ref.get("effective_stars") or 0.0))))
    activity_score = F.greatest(F.lit(0.0), F.lit(1.5) - F.abs(F.col("activity_intensity_score") - F.lit(float(ref.get("activity_intensity_score") or 0.0))))

    scored_df = (
        candidates_df
        .withColumn("shared_category_score", shared_category_score)
        .withColumn("city_score", city_score)
        .withColumn("state_score", state_score)
        .withColumn("price_score", price_score)
        .withColumn("stars_score", stars_score)
        .withColumn("activity_score", activity_score)
        .withColumn("similarity_score", F.col("shared_category_score") + F.col("city_score") + F.col("state_score") + F.col("price_score") + F.col("stars_score") + F.col("activity_score"))
        .filter(F.col("similarity_score") > 0)
        .select(
            "business_id", "business_name", "city", "state", "price_range_label",
            "effective_stars", "actual_review_count", "total_checkins", "tip_count",
            "similarity_score", "shared_category_score", "city_score", "state_score", "price_score", "stars_score", "activity_score"
        )
        .orderBy(F.desc("similarity_score"), F.desc("effective_stars"), F.desc("actual_review_count"))
        .limit(top_n)
    )

    return ref, scored_df


In [ ]:
reference_business_candidates_df = (
    content_recommendation_feature_df
    .filter(F.col("actual_review_count") >= 250)
    .select("business_id", "business_name", "city", "state", "actual_review_count")
    .orderBy(F.desc("actual_review_count"))
    .limit(10)
)

reference_business_candidates_df.show(truncate=False)


## Phase 20. Recommendation Demonstrations

This phase demonstrates the recommender using a few high-activity reference businesses from the enriched Part A output. The goal is to show how the recommendation logic behaves in practice rather than to present only abstract formulas.

In [ ]:
demo_reference_rows = reference_business_candidates_df.limit(3).collect()
for demo_row in demo_reference_rows:
    demo_ref, demo_recommendations_df = recommend_similar_businesses(demo_row["business_id"], top_n=5, same_state_only=True)
    print(f"Reference: {demo_ref['business_name']} ({demo_ref['city']}, {demo_ref['state']})")
    demo_recommendations_df.show(truncate=False)


In [ ]:
overall_popular_recommendations_df = recommend_popular_businesses(top_n=10, min_reviews=100)
overall_popular_recommendations_df.show(truncate=False)


## Phase 21. Baseline vs Main Method Discussion

The popularity baseline is useful for surfacing broadly strong businesses, but it mainly rewards businesses that already perform well on global quality and activity signals. The content-based recommender is more targeted because it starts from a reference business and then looks for relevant similarity in category, place, price, review quality, and activity profile. This makes the main method better aligned with practical business discovery, while the baseline remains a helpful benchmark.

## Phase 22. Part B Findings and Interpretation

The recommendation stage shows that the structured outputs from Part A can be reused directly for recommendation rather than only for reporting. The popularity baseline confirms which businesses look strongest from an overall quality-and-activity perspective, while the content-based recommender shows how those same enriched features can be repurposed into more context-sensitive recommendations. This means the project now completes the second half of the problem statement: the characteristics discovered through big data analytics are not only interpretable, but also operationally useful for generating relevant business recommendations.

### Ranking-Based Offline Evaluation

To make the evaluation stronger, I also use the user-business interaction matrix from the review data. I treat businesses with `rating >= 4` as positive interactions. Then for users who positively rated at least two businesses in the same state, I use one business as the **seed** and another as the **hidden relevant target**. After that, I run the content-based recommender and check whether the hidden target appears in the top recommendations.

This gives a more standard recommendation-style evaluation using ranking metrics such as **Hit Rate@5**, **Hit Rate@10**, and **MRR@10**.

In [ ]:
content_business_state_df = content_recommendation_feature_df.select(
    F.col("business_id").alias("content_business_id"),
    F.col("state").alias("content_state")
)

positive_user_business_df = (
    user_business_interactions_df
    .filter(F.col("rating") >= 4)
    .select("user_id", "business_id")
    .distinct()
)

offline_eval_case_pool_df = (
    positive_user_business_df
    .groupBy("user_id")
    .agg(F.sort_array(F.collect_set("business_id")).alias("positive_businesses"))
    .filter(F.size("positive_businesses") >= 2)
    .select(
        "user_id",
        F.element_at("positive_businesses", 1).alias("seed_business_id"),
        F.element_at("positive_businesses", 2).alias("target_business_id"),
        F.size("positive_businesses").alias("positive_business_count")
    )
    .join(
        content_business_state_df.withColumnRenamed("content_business_id", "seed_business_id").withColumnRenamed("content_state", "seed_state"),
        on="seed_business_id",
        how="inner"
    )
    .join(
        content_business_state_df.withColumnRenamed("content_business_id", "target_business_id").withColumnRenamed("content_state", "target_state"),
        on="target_business_id",
        how="inner"
    )
    .filter(F.col("seed_state") == F.col("target_state"))
)

offline_eval_case_pool_df.select(
    "user_id", "seed_business_id", "target_business_id", "seed_state", "positive_business_count"
).show(10, truncate=False)


In [ ]:
OFFLINE_EVAL_CASES = 100

offline_eval_cases = offline_eval_case_pool_df.limit(OFFLINE_EVAL_CASES).collect()
offline_eval_rows = []

for eval_case in offline_eval_cases:
    seed_business_id = eval_case["seed_business_id"]
    target_business_id = eval_case["target_business_id"]
    user_id = eval_case["user_id"]
    seed_state = eval_case["seed_state"]

    try:
        ref, recs_df = recommend_similar_businesses(seed_business_id, top_n=10, same_state_only=True)
        rec_business_ids = [row["business_id"] for row in recs_df.select("business_id").collect()]
        hit_at_5 = 1 if target_business_id in rec_business_ids[:5] else 0
        hit_at_10 = 1 if target_business_id in rec_business_ids[:10] else 0
        rank = rec_business_ids.index(target_business_id) + 1 if target_business_id in rec_business_ids else None
        mrr_at_10 = (1.0 / rank) if rank is not None and rank <= 10 else 0.0
        offline_eval_rows.append({
            "user_id": user_id,
            "seed_business_id": seed_business_id,
            "target_business_id": target_business_id,
            "seed_business_name": ref["business_name"],
            "seed_state": seed_state,
            "hit_at_5": hit_at_5,
            "hit_at_10": hit_at_10,
            "rank_of_target": rank,
            "mrr_at_10": mrr_at_10,
        })
    except Exception as exc:
        offline_eval_rows.append({
            "user_id": user_id,
            "seed_business_id": seed_business_id,
            "target_business_id": target_business_id,
            "seed_business_name": None,
            "seed_state": seed_state,
            "hit_at_5": 0,
            "hit_at_10": 0,
            "rank_of_target": None,
            "mrr_at_10": 0.0,
        })

offline_eval_results_df = pd.DataFrame(offline_eval_rows)
offline_eval_results_df.head(10)


In [ ]:
part_b_metric_evaluation_df = pd.DataFrame([
    {
        "metric": "Evaluation cases",
        "value": len(offline_eval_results_df),
    },
    {
        "metric": "Hit Rate@5",
        "value": round(float(offline_eval_results_df["hit_at_5"].mean()), 4) if len(offline_eval_results_df) > 0 else None,
    },
    {
        "metric": "Hit Rate@10",
        "value": round(float(offline_eval_results_df["hit_at_10"].mean()), 4) if len(offline_eval_results_df) > 0 else None,
    },
    {
        "metric": "MRR@10",
        "value": round(float(offline_eval_results_df["mrr_at_10"].mean()), 4) if len(offline_eval_results_df) > 0 else None,
    },
])

part_b_metric_evaluation_df


### What These Metrics Mean

- **Hit Rate@5** shows how often the hidden relevant business appeared in the top 5 recommendations.
- **Hit Rate@10** shows how often it appeared in the top 10 recommendations.
- **MRR@10** rewards cases where the relevant business appears closer to the top of the list.

These metrics do not mean the system is fully personalised, but they do give a more formal way to evaluate whether the similarity-based recommendations are recovering businesses that users also liked in practice.

## Recommendation Input Testing Block

The core Part B implementation above already demonstrates both the popularity baseline and the main content-based recommender. This optional block is added for practical testing and demo use: it allows a user to enter a business name and optional location filters, then generate recommendations directly from the notebook.

In [ ]:
def find_business_candidates_by_name(business_name, city=None, state=None, limit=10):
    df = content_recommendation_feature_df.filter(F.lower(F.col("business_name")) == business_name.lower())
    if city is not None:
        df = df.filter(F.lower(F.col("city")) == city.lower())
    if state is not None:
        df = df.filter(F.lower(F.col("state")) == state.lower())
    return df.select(
        "business_id", "business_name", "city", "state",
        "effective_stars", "actual_review_count", "total_checkins", "tip_count"
    ).orderBy(F.desc("actual_review_count")).limit(limit)

def run_content_recommendation_by_name(business_name, city=None, state=None, top_n=5, same_state_only=True):
    candidates = find_business_candidates_by_name(business_name, city=city, state=state, limit=10)
    candidate_rows = candidates.collect()

    if not candidate_rows:
        print("No matching business was found for the provided input.")
        return None

    if len(candidate_rows) > 1:
        print("Multiple matching businesses were found. Use one of these business_id values to narrow the test case:")
        candidates.show(truncate=False)
        return None

    selected = candidate_rows[0]
    ref, recommendations_df = recommend_similar_businesses(selected['business_id'], top_n=top_n, same_state_only=same_state_only)
    print(f"Reference business: {ref['business_name']} ({ref['city']}, {ref['state']})")
    recommendations_df.show(truncate=False)
    return recommendations_df

def run_popularity_recommendation_test(city=None, category=None, top_n=5, min_reviews=50):
    results_df = recommend_popular_businesses(city=city, category=category, top_n=top_n, min_reviews=min_reviews)
    print(f"Popularity baseline test -> city={city}, category={category}, top_n={top_n}, min_reviews={min_reviews}")
    results_df.show(truncate=False)
    return results_df


In [ ]:
# Manual test input cell for the main content-based recommender
TEST_BUSINESS_NAME = "Acme Oyster House"
TEST_CITY = "New Orleans"
TEST_STATE = "LA"
TEST_TOP_N = 5
TEST_SAME_STATE_ONLY = True

run_content_recommendation_by_name(
    business_name=TEST_BUSINESS_NAME,
    city=TEST_CITY,
    state=TEST_STATE,
    top_n=TEST_TOP_N,
    same_state_only=TEST_SAME_STATE_ONLY
)


In [ ]:
# Manual test input cell for the popularity baseline
BASELINE_TEST_CITY = "Philadelphia"
BASELINE_TEST_CATEGORY = None  # Example: "Restaurants"
BASELINE_TEST_TOP_N = 5
BASELINE_TEST_MIN_REVIEWS = 100

run_popularity_recommendation_test(
    city=BASELINE_TEST_CITY,
    category=BASELINE_TEST_CATEGORY,
    top_n=BASELINE_TEST_TOP_N,
    min_reviews=BASELINE_TEST_MIN_REVIEWS
)


**How to use this block in the demo**

1. Change `TEST_BUSINESS_NAME`, `TEST_CITY`, and `TEST_STATE` in the content-based test cell.
2. Run that cell to get similar business recommendations.
3. Change `BASELINE_TEST_CITY` or `BASELINE_TEST_CATEGORY` in the baseline test cell if you want to test the popularity recommender.
4. Use the `recommendation_test_cases_df` table as a quick set of examples if you do not want to think of test cases during the video.
